> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows three representative workflows that combine several modules of the toolkit.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Representative Workflows

This notebook shows short end-to-end workflows that combine several modules of CLABTOOLKIT, one for each kind of analysis: structural, diffusion MRI and tractography, and functional connectivity. Each workflow starts with the list of the modules it loads and why, and finishes with a table of the computation time of every step.

The workflows run on the sample data of clabtoolkit, the surfaces, cortical maps, annotations, parcellation and tractograms of the Colin27 average brain, retrieved with `get_sample_data_dir` (see `misctools`). The surfaces, the cortical maps and the tractograms are downloaded from [Zenodo](https://doi.org/10.5281/zenodo.23702131) the first time they are needed, and the annotations and the parcellation are shipped with the package. Everything is retrieved from the sample data except the functional time series of Workflow C, which is simulated on the grid of the retrieved parcellation, because the sample data has no functional image. FreeSurfer is not needed. Each code cell is self-contained and can be run on its own.

### Table of Contents

---

| Workflow | Analysis | Modules | Steps |
|---|---|---|---|
| [A](#table-of-contents) | Structural analysis: correcting a vertex-wise parcellation and computing regional statistics | `surfacetools`, `freesurfertools`, `morphometrytools`, `visualizationtools` | load data, correct, compute statistics, visualize |
| [B](#workflow-a-correcting-a-vertex-wise-parcellation) | Diffusion MRI and tractography: streamlines connecting a pair of regions | `tracttools`, `parcellationtools`, `visualizationtools` | load data, filter, save and check, visualize |
| [C](#workflow-b-streamlines-connecting-a-pair-of-regions) | Connectivity and network analysis: the hippocampus and the temporal lobe | `parcellationtools`, `imagetools`, `freesurfertools`, `connectivitytools` | prepare data, select regions, compute matrix, graph metrics |

---


## Workflow A. Correcting a vertex-wise parcellation

Structural analysis. Relabels small disconnected clusters of the Desikan annotation of the sample data, computes the regional cortical thickness from the corrected annotation, and shows the original and corrected annotations side by side.

**Steps**

1. **Load the data:** the inflated surface, the cortical thickness map and the Desikan annotation.
2. **Correct the parcellation:** relabel small disconnected clusters on a copy of the annotation.
3. **Compute statistics:** regional cortical thickness from the corrected annotation.
4. **Save the corrected annotation:** save the annotation to a temporary folder
5. **Visualize:** the original and corrected annotations side by side.

In [1]:
# Workflow A: Correcting a vertex-wise parcellation and computing regional statistics

# Correcting a cortical annotation (the Desikan annotation of the sample data) and computing regional cortical thickness

# CLABTOOLKIT modules used in this example and why they are loaded:
#   - surfacetools:      provides the `Surface` object. It holds the cortical geometry, the
#                        vertex-wise thickness map and the annotations that are visualized.
#   - freesurfertools:   provides the `AnnotParcellation` object. It reads the annotation and
#                        implements `correct_vertexwise_parcellation`.
#   - misctools:         provides `get_sample_data_dir`, which finds (and downloads the first time)
#                        the sample data of clabtoolkit.
#   - morphometrytools:  provides `compute_reg_val_fromannot`, which summarizes a vertex-wise
#                        map (here, cortical thickness) within the regions of an annotation.
#   - visualizationtools: provides `BrainPlotter`, used to render the original and the corrected
#                        annotations side by side.

import copy
import os
import tempfile
import time

import numpy as np

import itables  # Only needed to display the tables (pip install "clabtoolkit[examples]")

import clabtoolkit.surfacetools as cltsurf
import clabtoolkit.freesurfertools as cltfree
import clabtoolkit.morphometrytools as cltmorpho
import clabtoolkit.visualizationtools as cltvis
import clabtoolkit.misctools as cltmisc

timings = {}  # Computation time (in seconds) of each step


def timed(label):
    """Context manager that stores the elapsed time of a block in `timings`."""

    class _Timer:
        def __enter__(self):
            self.t0 = time.perf_counter()

        def __exit__(self, *exc):
            timings[label] = time.perf_counter() - self.t0

    return _Timer()


# ----------------- Step 1: Preparing the needed data for the analysis (This step is not necessary if you already have the data)
# Sample data of clabtoolkit (the surface and the map are downloaded from Zenodo the first time; the annotation is shipped with the package)
hemi_id = "lh"
prefix = "sub-colin27_ses-01_hemi-L_space-tallin"
surf_file = os.path.join(cltmisc.get_sample_data_dir("surfaces"), f"{prefix}_inflated.surf.gii")
annot_file = os.path.join(cltmisc.get_sample_data_dir("annotations"), f"{prefix}_atlas-Desikan2006_dseg.label.gii")
thickness_file = os.path.join(cltmisc.get_sample_data_dir("corticalmaps"), f"{prefix}_thickness.shape.gii")

with timed("Step 1a. Load the surface and the thickness map"):
    surf = cltsurf.Surface(surf_file)  # Load the surface geometry
    surf.load_scalar_maps(thickness_file, maps_names="Thickness")  # Load the vertex-wise cortical thickness

with timed("Step 1b. Load the original annotation"):
    annot_parc = cltfree.AnnotParcellation(annot_file, annot_id="Desikan")
    surf.load_annotation(annot_parc, parc_name="Desikan")

# The sample data has no cortex label: it is derived from the annotation (every vertex outside the medial wall, code 0)
codes = np.asarray(annot_parc.codes).ravel()
labeled_indices = np.where(codes != 0)[0]
label_data = np.concatenate([labeled_indices[:, None], surf.mesh.points[labeled_indices]], axis=1)
label_file = os.path.join(tempfile.gettempdir(), f"{hemi_id}.cortex.label")
with open(label_file, "w") as f:
    f.write("#!ascii label\n")
    f.write(f"{len(label_data)}\n")
    for idx, x, y, z in label_data:
        f.write(f"{int(idx)} {x:.6f} {y:.6f} {z:.6f} 0.500000\n")
# ----------------- End of Step 1 ---------------------------------------------------

# ----------------- Step 2: Correcting the vertex-wise parcellation
# `correct_vertexwise_parcellation` modifies the annotation in place, so we work on a copy.
# The surface is passed as a Surface object because the filling step needs a FreeSurfer-format surface file, which the method creates itself
with timed("Step 2. Correct the vertex-wise parcellation"):
    annot_corrected = copy.deepcopy(annot_parc)
    annot_corrected.correct_vertexwise_parcellation(
        surf, cortex_file=label_file, min_cluster_size=100
    )
    surf.load_annotation(annot_corrected, parc_name="Desikan_Corrected")

n_changed = int((annot_corrected.codes != annot_parc.codes).sum())
print(f"Vertices relabeled: {n_changed}")
# ----------------- End of Step 2 ---------------------------------------------------

# ----------------- Step 3: Computing regional statistics on the corrected annotation
with timed("Step 3. Compute the regional cortical thickness"):
    df_region_thickness, _, _ = cltmorpho.compute_reg_val_fromannot(
        metric_file=thickness_file,
        parc_file=annot_corrected,
        hemi=hemi_id,
        metric="Thickness",
        include_global=False,
        add_bids_entities=False,
    )
itables.show(df_region_thickness)
# ----------------- End of Step 3 ---------------------------------------------------

# ----------------- Step 4: Saving the corrected annotation
with timed("Step 4. Save the corrected annotation"):
    corrected_annot_file = os.path.join(tempfile.gettempdir(), f"{hemi_id}.Desikan_Corrected.annot")
    annot_corrected.save_annotation(corrected_annot_file, overwrite=True)
    print(f"Corrected annotation saved to: {corrected_annot_file}")
# ----------------- End of Step 4 ---------------------------------------------------

# ----------------- Step 5: Visualizing the original and corrected annotations side by side
with timed("Step 4. Plot the original and corrected annotations"):
    plotter = cltvis.BrainPlotter()
    plotter.plot(
        objs2plot=surf,
        views="medial",
        map_names=["Desikan", "Desikan_Corrected"], 
        use_opacity=False
    )
# ----------------- End of Step 4 ---------------------------------------------------

# ----------------- Computation times
time_total = sum(timings.values())

print(f"\n{'Step':<56s}{'Time (s)':>10s}")
print("-" * 66)
for step, seconds in timings.items():
    print(f"{step:<56s}{seconds:>10.2f}")
print("-" * 66)
print(f"{'Total':<56s}{time_total:>10.2f}")

Vertices relabeled: 25


Loading ITables v2.8.0 from the internet... (need help?)


Corrected annotation saved to: /tmp/lh.Desikan_Corrected.annot
Number of views: 1, Number of maps: 2, Number of objects: 1

Step                                                      Time (s)
------------------------------------------------------------------
Step 1a. Load the surface and the thickness map               0.07
Step 1b. Load the original annotation                         0.03
Step 2. Correct the vertex-wise parcellation                  1.48
Step 3. Compute the regional cortical thickness               0.02
Step 4. Save the corrected annotation                         0.02
Step 4. Plot the original and corrected annotations           4.19
------------------------------------------------------------------
Total                                                         5.83



## Workflow B. Streamlines connecting a pair of regions

Diffusion MRI and tractography. Selects the callosal streamlines that connect the left and right precentral cortex, saves them in TRK and TCK format and checks the saved files.

**Steps**

1. **Load the data:** the sample tractogram and the sample parcellation.
2. **Select the streamlines:** keep those that connect the left and right superior frontal gyri.
3. **Save and check:** write TRK and TCK files, reload them and compare streamline counts and coordinates.
4. **Visualize:** the original and filtered tractograms, colored by region labels and by length.

In [ ]:
# Workflow B: Selecting the streamlines that connect a pair of cortical regions

# Extracting the callosal streamlines that connect the left and right precentral cortex

# CLABTOOLKIT modules used in this example and why they are loaded:
#   - tracttools:         provides the `Tractogram` object. It reads the TRK file, computes the
#                         streamline lengths, selects the streamlines that connect a pair of regions
#                         (`filter_by_mask`) and saves the result in TRK and TCK format.
#   - parcellationtools:  provides the `Parcellation` object. It reads the sample parcellation
#                         and its color table, which define the regions that are connected.
#   - misctools:          provides `get_sample_data_dir`, which finds (and downloads the first time)
#                         the sample data of clabtoolkit.
#   - visualizationtools: provides `BrainPlotter`, used to render the original and the filtered
#                         tractograms side by side.

import os
import tempfile
import time
from pathlib import Path

import numpy as np

import clabtoolkit.tracttools as clttract
import clabtoolkit.parcellationtools as cltparc
import clabtoolkit.visualizationtools as cltvis
import clabtoolkit.misctools as cltmisc

timings = {}  # Computation time (in seconds) of each step


def timed(label):
    """Context manager that stores the elapsed time of a block in `timings`."""

    class _Timer:
        def __enter__(self):
            self.t0 = time.perf_counter()

        def __exit__(self, *exc):
            timings[label] = time.perf_counter() - self.t0

    return _Timer()


# ----------------- Step 1: Preparing the needed data for the analysis (This step is not necessary if you already have the data)
# Sample data of clabtoolkit (the tractogram is downloaded from Zenodo the first time; the parcellation is shipped with the package)
in_trk = os.path.join(cltmisc.get_sample_data_dir("tractograms"), "sub-colin27_ses-01_space-tallin_label-cc_tractogram.trk")
parc_dir = cltmisc.get_sample_data_dir("parcellations")
parc_file = os.path.join(parc_dir, "sub-colin27_ses-01_space-tallin_atlas-chimeraDFMFHISIFN_seg-aparc_desc-grow1mm_dseg.nii.gz")
lut_file = os.path.join(parc_dir, "sub-colin27_ses-01_space-tallin_atlas-chimeraDFMFHISIFN_seg-aparc_desc-grow1mm_dseg.lut")

with timed("Step 1a. Load the tractogram"):
    tract_obj = clttract.Tractogram(in_trk)
    tract_obj.compute_streamline_lengths()  # Stored in tract_obj.data_per_streamline["length"]

with timed("Step 1b. Load the parcellation"):
    parc = cltparc.Parcellation(parc_file, color_table=lut_file)

print(f"Number of streamlines in the original tractogram: {len(tract_obj.tracts)}")
# ----------------- End of Step 1 ---------------------------------------------------

# ----------------- Step 2: Selecting the streamlines that connect the regions of interest
# Labels of the left and right precentral cortex, looked up by name in the sample parcellation
reg_dict = {parc.names_to_labels("ctx-lh-superiorfrontal")[0]: parc.names_to_labels("ctx-rh-superiorfrontal")[0]}

with timed("Step 2. Filter the tractogram by the pair of regions"):
    filtered, bundles, registry = tract_obj.filter_by_mask(
        parc, reg_dict, split_bundles=True
    )
    filtered.compute_streamline_lengths()

print(f"Number of streamlines kept: {len(filtered.tracts)}")
for pair_id, info in registry.items():
    print(f"  Pair {pair_id}: {info['name']} ({info['n_streamlines']} streamlines)")

len_all = tract_obj.data_per_streamline["length"].ravel()
len_kept = filtered.data_per_streamline["length"].ravel()
print(f"Mean length (original): {len_all.mean():.1f} mm")
print(f"Mean length (filtered): {len_kept.mean():.1f} mm")
# ----------------- End of Step 2 ---------------------------------------------------

# ----------------- Step 3: Saving the filtered tractogram and checking the saved files
out_dir = Path(tempfile.gettempdir())
with timed("Step 3. Save (TRK, TCK) and reload the tractogram"):
    for ext in ("trk", "tck"):
        out_file = str(out_dir / f"filtered_precentral.{ext}")
        filtered.save_tractogram(out_file, file_type=ext, overwrite=True)
        reloaded = clttract.Tractogram(out_file)
        same_n = len(reloaded.tracts) == len(filtered.tracts)
        max_diff = max(
            float(np.max(np.abs(a - b))) for a, b in zip(filtered.tracts, reloaded.tracts)
        )
        print(f"  {ext.upper()}: same number of streamlines = {same_n}, "
            f"max coordinate difference = {max_diff:.2e} mm")
# ----------------- End of Step 3 ---------------------------------------------------

# ----------------- Step 4: Visualizing the original and the filtered tractograms
with timed("Step 4. Plot the original and the filtered tractograms"):

    # Keep only the superiorfrontal regions in the parcellation and interpolate the bundle IDs on the tractograms
    parc.keep_by_name(["ctx-lh-superiorfrontal", "ctx-rh-superiorfrontal"])
    tract_obj.interpolate_on_tractogram(parc, map_name="region_labels") # Interpolating the region labels for the original tractogram
    filtered.interpolate_on_tractogram(parc, map_name="region_labels") # Interpolating the region labels for the filtered tractogram

    plotter = cltvis.BrainPlotter()
    plotter.plot(
        [tract_obj, filtered],                                # Showing the original and the filtered tractograms (Rows)
        map_names=["interp_mask", "region_labels", "length"], # Showing the three maps (Columns): the interception between parcellation and tractogram, 
        use_opacity=False,                                    # the interception between the tractogram and the selected region labels, and the streamline length
        colorbar_titles="Length (mm)"
    )
# ----------------- End of Step 4 ---------------------------------------------------

# ----------------- Computation times
time_total = sum(timings.values())

print(f"\n{'Step':<56s}{'Time (s)':>10s}")
print("-" * 66)
for step, seconds in timings.items():
    print(f"{step:<56s}{seconds:>10.2f}")
print("-" * 66)
print(f"{'Total':<56s}{time_total:>10.2f}")


## Workflow C. Connectivity of the hippocampus and the temporal lobe

Connectivity and network analysis. Retrieves the sample parcellation and simulates a functional time series on its grid, because the sample data has no functional image. It then computes the functional connectivity matrix of the hippocampus and the temporal lobe, thresholds it and calculates graph metrics. The times are reported with and without the simulation of the time series.

**Steps**

1. **Prepare the data:** retrieve the sample parcellation and simulate the functional time series on its grid (not needed with real data).
2. **Select the regions:** temporal-lobe cortex and hippocampus.
3. **Compute and plot** the functional connectivity matrix.
4. **Threshold and compute graph metrics** of the connectivity matrix.

In [ ]:
# Workflow C: Functional connectivity analysis

# Connectivity analysis of the hippocampus and the temporal lobe

# CLABTOOLKIT modules used in this example and why they are loaded:
#   - parcellationtools:  provides the `Parcellation` object. It reads the label image and computes
#                         the region-wise time series and the functional connectivity matrix.
#   - imagetools:         provides `simulate_image`, used to simulate the functional time series on
#                         the grid of the retrieved parcellation (not needed with real data).
#   - freesurfertools:    provides `load_lobes_json`, which gives the Desikan region names grouped
#                         by lobe, used here to select the temporal lobe.
#   - misctools:          provides `get_sample_data_dir`, which finds the sample parcellation, and
#                         `filter_by_substring`, used to select the regions of interest by name.
# The `Connectome` object returned by `compute_fc_matrix` comes from connectivitytools, which is
# loaded automatically. It provides the plotting, thresholding and graph metrics of Steps 3 and 4.

import os
import time
import clabtoolkit.parcellationtools as cltparc
import clabtoolkit.imagetools as cltimg
import clabtoolkit.freesurfertools as cltfree
import clabtoolkit.misctools as cltmisc
from pathlib import Path
import tempfile
import itables  # Only needed to display the tables (pip install "clabtoolkit[examples]")

timings = {}  # Computation time (in seconds) of each step


def timed(label):
    """Context manager that stores the elapsed time of a block in `timings`."""

    class _Timer:
        def __enter__(self):
            self.t0 = time.perf_counter()

        def __exit__(self, *exc):
            timings[label] = time.perf_counter() - self.t0

    return _Timer()


# ----------------- Step 1: Preparing the needed data for the analysis (This step is not necessary if you already have the data)
# Sample data of clabtoolkit: the parcellation is retrieved (it is shipped with the package)
parc_file = os.path.join(cltmisc.get_sample_data_dir("parcellations"), "sub-colin27_ses-01_space-tallin_atlas-chimeraDFMFHISIFN_seg-aparc_desc-grow1mm_dseg.nii.gz")

# Load the parcellation
with timed("Step 1a. Load the parcellation"):
    parc = cltparc.Parcellation(parc_file)

# The sample data has no functional image, so some preprocessed functional data are simulated
# in the space defined by the retrieved parcellation
# (random_seed makes the simulation reproducible)
sim_func_image = str(Path(tempfile.gettempdir()) / "simulated_preproc_func.nii.gz")
with timed("Step 1b. Simulate the functional data"):
    func_data = cltimg.simulate_image(
        parc, n_volumes=10, simulated_image=sim_func_image, random_seed=42
    )
# ----------------- End of Step 1 ---------------------------------------------------

# ----------------- Step 2: Getting the regions of interest for the connectivity analysis
with timed("Step 2. Select the regions of interest"):
    # Temporal lobe regions of the Desikan atlas
    ctx_desikan_temporal = cltfree.load_lobes_json()["desikan"]["lobes"]["temporal"]

    # Keeping only the regions with the ctx prefix (gm), to avoid the cortical white matter (wm-)
    # regions, and only those belonging to the temporal lobe
    reg_names = cltmisc.filter_by_substring(
        parc.name, and_filter="ctx-", or_filter=ctx_desikan_temporal
    )
    reg_names = reg_names + ["hipp-"]  # Adding the hippocampus regions
# ----------------- End of Step 2 ---------------------------------------------------

# ----------------- Step 3: Computing and plotting the connectivity matrix
with timed("Step 3a. Compute the functional connectivity matrix"):
    connectivity_matrix = parc.compute_fc_matrix(
        sim_func_image,
        vols_to_delete="1-3",  # Deleting the first 3 volumes from the time series
        region_names=reg_names,  # Specifying the regions of interest for the connectivity analysis
        ts_method="clabtoolkit",  # Using clabtoolkit method for time series extraction
    )

with timed("Step 3b. Plot the connectivity matrix"):
    connectivity_matrix.plot_matrix(figsize=(7, 7))
# ----------------- End of Step 3 ---------------------------------------------------

# ----------------- Step 4: Compute graph metrics of the connectivity matrix
# The functional data are simulated noise, so the correlations are close to zero and a fixed value
# threshold would leave almost no connections. We keep the strongest 20% instead.
with timed("Step 4a. Threshold and binarize the matrix"):
    connectome_thr = connectivity_matrix.threshold(
        method="sparsity", threshold=0.2, binarize=True, absolute=True
    )

with timed("Step 4b. Compute the graph metrics"):
    metrics = connectome_thr.get_graph_metrics()  # Returns a single DataFrame
itables.show(metrics)
# ----------------- End of Step 4 ---------------------------------------------------

# ----------------- Computation times
time_sim = timings["Step 1b. Simulate the functional data"]  # Only needed because the data are simulated
time_total = sum(timings.values())

print(f"\n{'Step':<56s}{'Time (s)':>10s}")
print("-" * 66)
for step, seconds in timings.items():
    print(f"{step:<56s}{seconds:>10.2f}")
print("-" * 66)
print(f"{'Total (with data simulation)':<56s}{time_total:>10.2f}")
print(f"{'Total (without data simulation)':<56s}{time_total - time_sim:>10.2f}")